In [1]:
# Setup sys.path for notebook imports
from pathlib import Path
import sys

p = Path.cwd()
for _ in range(6):
	if (p / 'backend').exists():
		sys.path.insert(0, str(p))
		break
	p = p.parent
print('PYTHONPATH root:', sys.path[0])

PYTHONPATH root: c:\Users\marce\projetos\quant-terminal


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go

from backend.data_pipeline.extract.bcb import get_selic_atual, get_datas_copom
from backend.data_pipeline.extract.infomoney import get_di_futuros_from_infomoney

URL_DATAS_COPOM_2025 = "https://www.bcb.gov.br/api/conteudo/app/normativos/exibeoutrasnormas?p1=COMUNICADO&p2=41779"
URL_DATAS_COPOM_2026 = "https://www.bcb.gov.br/api/conteudo/app/normativos/exibeoutrasnormas?p1=COMUNICADO&p2=43383"

In [3]:
data_referencia = pd.to_datetime('today').date() # BZDIOVRA 
idi_hoje = 453365.72                             # IDIX9

selic, data = get_selic_atual()
print(f"Selic atual ({data}): {selic:.2f}%")

Selic atual (01/08/2025): 14.90%


In [7]:
# Depois tenar pegar curva de juros da B3 ao invés da Infomoney
# https://www2.bmf.com.br/pages/portal/bmfbovespa/lumis/lum-boletim-online-ptBR.asp?Acao=BUSCA&cboMercadoria=DI1

curva_di_fut = get_di_futuros_from_infomoney(pd.to_datetime('today').date())
print(curva_di_fut)

# URLs dos comunicados
urls = [ URL_DATAS_COPOM_2025, URL_DATAS_COPOM_2026 ]

# Extrai e junta todas as datas
reunioes_copom = []
for url in urls:
    reunioes_copom.extend(get_datas_copom(url))

# Ordena cronologicamente e filtra apenas próximas reuniões
reunioes_copom = [data for data in reunioes_copom if data > data_referencia]
reunioes_copom = sorted(reunioes_copom)

# Exibe
print(curva_di_fut)
print('-'*50)
print(reunioes_copom)

INFO:backend.data_pipeline.extract.infomoney:Iniciando extração de DI Futuro da InfoMoney para data: 2025-08-14
INFO:backend.data_pipeline.extract.infomoney:Fazendo requisição POST para InfoMoney
ERROR:backend.data_pipeline.extract.infomoney:Resposta vazia da InfoMoney


None
None
--------------------------------------------------
[datetime.date(2025, 9, 17), datetime.date(2025, 11, 5), datetime.date(2025, 12, 10), datetime.date(2026, 1, 28), datetime.date(2026, 3, 18), datetime.date(2026, 4, 29), datetime.date(2026, 8, 5), datetime.date(2026, 9, 16), datetime.date(2026, 11, 4), datetime.date(2026, 12, 9)]


In [ ]:
# Funções auxiliares para cálculo
def calcular_cdi_acumulado(taxa, dias_uteis):
    return (1 + taxa) ** (dias_uteis / 252)

def calcular_taxa_impl(n0, n1, taxa0, taxa1):
    # CDI acumulado (bruto)
    acc0 = calcular_cdi_acumulado(taxa0, n0)
    acc1 = calcular_cdi_acumulado(taxa1, n1)

    # CDI médio implícito entre as datas
    taxa_impl = (acc1 / acc0) ** (1 / ((n1 - n0)/252)) - 1
    return taxa_impl

# Calcular o IDI Projetado
def calcular_idi_proj(base_idi, taxa, dias_uteis):
    return base_idi * ((1 + taxa) ** (dias_uteis / 252))

# Calcula IDI projetado
curva_di_fut["IDI Projetado"] = curva_di_fut.apply(
    lambda row: calcular_idi_proj(idi_hoje, row["Taxa (%)"]/100, row["DU"]),
    axis=1
)

# Mostra resultado
print(curva_di_fut[["Vencimento", "DU", "Taxa (%)", "IDI Projetado"]].head())

In [ ]:
# Gerar DataFrame com todos os eventos
eventos = []
vencimentos = sorted(list(curva_di_fut["Vencimento"]))
vencimentos_dt = [pd.Timestamp(d) for d in vencimentos]

curva_di_ate_ult_reuniao_copom = curva_di_fut[curva_di_fut["Vencimento"] <= max(reunioes_copom)]

# Iterar pelos pares de vértices para estimar variação implícita
for i in range(len(curva_di_ate_ult_reuniao_copom) - 1):
    row0 = curva_di_ate_ult_reuniao_copom.iloc[i]
    row1 = curva_di_ate_ult_reuniao_copom.iloc[i + 1]

    n0 = int(row0["DU"])
    n1 = int(row1["DU"])
    taxa0 = row0["Taxa (%)"]/100
    taxa1 = row1["Taxa (%)"]/100

    if i==0:
        estimativa_impl = calcular_taxa_impl(0, n0, selic/100, taxa1)
        eventos.append({
            "Data": row0["Vencimento"],
            "Evento": f"Vencimento DI ({row0["Contrato"]})",
            "Vencimento DI": row0["Vencimento"].strftime("%d/%m/%Y"),
            "DU": row0["DU"],
            "Last": f"{taxa0:.3%}",
            "Est": f"{estimativa_impl:.3%}",
            "Last - Est": f"{taxa0 - estimativa_impl:.3%}",
            "COPOM Est": "",
            "COPOM + CDI": "",
            "IDI Est": f"{row0['IDI Projetado']:.3f}"
        })
    else:
        estimativa_impl = calcular_taxa_impl(n0, n1, taxa0, taxa1)
        eventos.append({
            "Data": row1["Vencimento"],
            "Evento": f"Vencimento DI ({row1["Contrato"]})",
            "Vencimento DI": row1["Vencimento"].strftime("%d/%m/%Y"),
            "DU": row1["DU"],
            "Last": f"{taxa1:.3%}",
            "Est": f"{estimativa_impl:.3%}",
            "Last - Est": f"{taxa1 - estimativa_impl:.3%}",
            "COPOM Est": "",
            "COPOM + CDI": "",
            "IDI Est": f"{row1['IDI Projetado']:.3f}"
        })

# Adicionar eventos do COPOM
for r in reunioes_copom:
    eventos.append({
        "Data": r,
        "Evento": "Reunião do COPOM",
        "Vencimento DI": "",
        "DU": "",
        "Last": "",
        "Est": "",  # pode ser preenchida com taxa interpolada
        "Last - Est": "",
        "COPOM Est": "",
        "COPOM + CDI": "",
        "IDI Est": ""
    })

df_eventos = pd.DataFrame(eventos)
df_eventos.sort_values(by="Data", inplace=True)
df_eventos.reset_index(drop=True, inplace=True)

# Exibir as 10 primeiras linhas
print(df_eventos.head(10))

In [ ]:
# Gráfico Histórico (Opção 1 - Plotly)
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_eventos['Data'],
    y=df_eventos['estimativa_selic'],
    mode='lines+markers',
    name='Expectativa Selic'
))

fig.update_layout(
    title='Histórico de Expectativa de Selic para a Próxima Reunião do COPOM',
    xaxis_title='Data de Observação',
    yaxis_title='Taxa Selic Projetada (%)',
    template='plotly_white'
)

fig.show()

In [ ]:
# Gráfico Histórico (Opção 2 - Matplotlib)

plt.figure(figsize=(10, 6))
plt.plot(df_eventos['Data'], df_eventos['estimativa_selic'], marker='o')
plt.title('Histórico da Expectativa de Selic para a Próxima Reunião do COPOM')
plt.xlabel('Data de Observação')
plt.ylabel('Taxa Selic Projetada (%)')
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()